In [1]:
!pip install pandas
!pip install matplotlib

In [4]:
import pandas as pd
import glob

# Training Data
Gathering necessary data from the each heuristic's training run:

In [9]:
vanilla_files = glob.glob('../results/vanilla_alpha_zero_28/logs/go/9x9/actor*.csv')
vanilla_combined = pd.concat([pd.read_csv(f) for f in vanilla_files], ignore_index=True)

vanilla_combined["datetime"] = pd.to_datetime(vanilla_combined["datetime"])
vanilla_combined = vanilla_combined.sort_values("datetime")

n_games = vanilla_combined.shape[0]
n_states = vanilla_combined["game_length"].sum()
n_sim = n_states * 200
sim_p_situation = round(n_sim / n_states, 2)
#median_sim = vanilla_combined["game_length"].median() * 200
vanilla_data = [n_games, n_states, n_sim, sim_p_situation]

In [10]:
topk_files = glob.glob('../results/top_k_alpha_zero_28/logs/go/9x9/actor*.csv')
topk_combined = pd.concat([pd.read_csv(f) for f in topk_files], ignore_index=True)

topk_combined["datetime"] = pd.to_datetime(topk_combined["datetime"])
topk_combined = topk_combined.sort_values("datetime")

n_games = int(topk_combined.shape[0])
n_states = int(topk_combined["game_length"].sum())
n_sim = int(topk_combined["game_simulations"].sum())
sim_p_situation = round(n_sim / n_states, 2)
#median_sim = topk_combined["game_simulations"].median()
topk_data = [n_games, n_states, n_sim, sim_p_situation]

In [11]:
topbotk_files = glob.glob('../results/top_bot_k_alpha_zero_28/logs/go/9x9/actor*.csv')
topbotk_combined = pd.concat([pd.read_csv(f) for f in topbotk_files], ignore_index=True)

topbotk_combined["datetime"] = pd.to_datetime(topbotk_combined["datetime"])
topbotk_combined = topbotk_combined.sort_values("datetime")

n_games = int(topbotk_combined.shape[0])
n_states = int(topbotk_combined["game_length"].sum())
n_sim = int(topbotk_combined["game_simulations"].sum())
sim_p_situation = round(n_sim / n_states, 2)
#median_sim = topbotk_combined["game_simulations"].median()
topbotk_data = [n_games, n_states, n_sim, sim_p_situation]

In [12]:
rl_files = glob.glob('../results/rl_alpha_zero_28/logs/go/9x9/actor*.csv')
rl_combined = pd.concat([pd.read_csv(f) for f in rl_files], ignore_index=True)

rl_combined["datetime"] = pd.to_datetime(rl_combined["datetime"])
rl_combined = rl_combined.sort_values("datetime")

n_games = int(rl_combined.shape[0])
n_states = int(rl_combined["game_length"].sum())
n_sim = int(rl_combined["game_simulations"].sum())
sim_p_situation = round(n_sim / n_states, 2)
#median_sim = rl_combined["game_simulations"].median()
rl_data = [n_games, n_states, n_sim, sim_p_situation]

In [15]:
rand_files = glob.glob('../results/rand_alpha_zero_28/logs/go/9x9/actor*.csv')
rand_combined = pd.concat([pd.read_csv(f) for f in rand_files], ignore_index=True)

rand_combined["datetime"] = pd.to_datetime(rand_combined["datetime"])
rand_combined = rand_combined.sort_values("datetime")

n_games = int(rand_combined.shape[0])
n_states = int(rand_combined["game_length"].sum())
n_sim = int(rand_combined["game_simulations"].sum())
sim_p_situation = round(n_sim / n_states, 2)
#median_sim = rand_combined["game_simulations"].median()
rand_data = [n_games, n_states, n_sim, sim_p_situation]

In [16]:
training_data = pd.DataFrame({
    'vanilla_az': vanilla_data,
    'top_k_az': topk_data,
    'top_bot_k_az': topbotk_data,
    'rl_az': rl_data,
    'rand_az': rand_data
})

training_data.to_csv("training_data.csv", index=False)

# Game Data
Gathering necessary data from the test games:

In [13]:
import re
def get_wins(file_name):
    file = "../get_results/"+file_name+"/log.csv"
    file_csv = pd.read_csv(file)
    wins_black = 0
    wins_white = 0
    black_sims = 0
    white_sims = 0
    for _, row in file_csv.iterrows():
        black_sims += row["black_sim"]
        white_sims += row["white_sim"]
        game_length += round(row["game_length"] / 2)
        if re.match(r"B\+", row["game_result"]):
            wins_black+=1    
        else:
            wins_white+=1
        
    return wins_black, wins_white, black_sims, white_sims

In [56]:
models = ["vanilla_az", "top_k_az", "top_bot_k_az", "rl_az", "rand_az"]
game_results_black = pd.DataFrame(index=models, columns=models)
game_results_white = pd.DataFrame(index=models, columns=models)
game_results_total = pd.DataFrame(index=models)
game_sims_black = pd.DataFrame(index=models, columns=models)
game_sims_white = pd.DataFrame(index=models, columns=models)


files = glob.glob("../get_results/**/log.csv", recursive=True)
games = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

for i, model1 in enumerate(models):
    for j, model2 in enumerate(models):
        if i == j:
            game_results_black.loc[model1, model2] = 0
            game_results_white.loc[model1, model2] = 0
            game_sims_black.loc[model1, model2] = 0
            game_sims_white.loc[model1, model2] = 0
        else:
            game_results_black.loc[model1, model2], game_results_white.loc[model2, model1], game_sims_black.loc[model1, model2], game_sims_white.loc[model1, model2] = get_wins(str(i)+"x"+str(j)+"_matches")

wins_as_black = game_results_black.sum(axis=1)
wins_as_white = game_results_white.sum(axis=1)

game_results_total = wins_as_black + wins_as_white

game_results_opp = game_results_black + game_results_white

sims_as_black = game_sims_black.sum(axis=1)
sims_as_white = game_sims_white.sum(axis=1)

game_sims_total = sims_as_black + sims_as_white
game_sims_total = game_sims_total.round()
game_results_black.to_csv("games_data_black.csv", index=False)
game_results_white.to_csv("games_data_white.csv", index=False)
game_results_opp.to_csv("game_results_opp.csv", index=False)
game_results_total.to_csv("games_data_total.csv", index=False)
game_sims_black.to_csv("game_sims_black.csv", index=False)
game_sims_white.to_csv("game_sims_white.csv", index=False)
game_sims_total.to_csv("games_sims_total.csv", index=False)